# 05_genomic_prep — Genomic prep (ACAF subset/QC/export) + positive-control GWAS
**Session 4 · GWAS of Maladaptive Recovery After Elective Orthopedic Surgery (All of Us / Verily Workbench 2.0)**

## Cost headline (§ COST PROTOCOL)
The Hail/Dataproc Spark cluster is the single most expensive item in the project. **This session avoids it entirely.**
Recon showed the srWGS **ACAF** callset ships as per-chromosome PLINK2 `pgen` + PLINK1 `bed` + `bgen` (not only Hail MT/VDS),
so the cohort subset + QC + per-ancestry export are done with **plink2 on one cheap high-mem VM** (AoU Jupyter, n1-highmem-4, ~$0.25/hr).
REGENIE (pre-installed) runs on the same VM. No cluster was launched.

## Compliance (DUCC §3.1)
Aggregate-only, round-to-20, GWAS **MAC≥20**; per-ancestry MAF≥1% guarantees per-variant MAC well above 20.
No individual-level data (person_ids, genotypes) is ever printed — keep-lists / genotype files stay in-VM or the workspace bucket.
Summary statistics stay in the workspace bucket until AoU dissemination review.


## 0. Config — resolve env, mount buckets (recreated in-VM each session)

In [ ]:
import os, subprocess
GOOGLE_PROJECT = os.environ.get("GOOGLE_PROJECT") or os.environ.get("GOOGLE_CLOUD_PROJECT")
def wb_resolve(name):
    for args in (["wb","resource","resolve","--name",name],["wb","resolve","--name",name]):
        try:
            out = subprocess.run(args, capture_output=True, text=True, timeout=90)
            if (out.stdout or "").strip(): return out.stdout.strip()
        except Exception: pass
    return None
WORKSPACE_CDR = os.environ.get("WORKSPACE_CDR") or wb_resolve("C2025Q4R6")          # <WORKSPACE_CDR>
WSB           = wb_resolve("gwas_analytic_bucket")                                   # <WORKSPACE_BUCKET>
GENOMICS      = wb_resolve("v9-genomics-folder")                                     # gs://vwb-aou-datasets-controlled/v9
print("PROJECT:", GOOGLE_PROJECT, "| CDR:", WORKSPACE_CDR, "| WSB:", WSB)
# Mount referenced buckets into the app (gcsfuse); ACAF is read directly from the mount.
subprocess.run(["wb","resource","mount"], check=False)
MNT  = os.path.expanduser("~/workspace/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold")
AB   = os.path.expanduser("~/workspace/gwas_analytic_bucket")
def r20(n):
    n=int(n); return 0 if n==0 else ("<=20 (suppressed)" if n<=20 else int(round(n/20.0)*20))


## 1. Locate + confirm the ACAF format (recon)
`gsutil ls .../acaf_threshold/` -> `bed/  bgen/  multiMT/  pgen/  plink_bed/  splitMT/  vcf/`.
PLINK2 `pgen` is ~10x smaller than PLINK1 `bed` (chr1: 116 GiB pgen vs 1.2 TiB bed) -> use **pgen** as the source.
Full callset = **535,663 samples**; `.psam` header is `#IID` (IID = person_id), `.pvar` is `#CHROM POS ID REF ALT INFO`.

In [ ]:
# confirm format + sizes (aggregate metadata only)
!gsutil -u $GOOGLE_PROJECT ls {MNT}/                     # bed/ bgen/ pgen/ plink_bed/ splitMT/ ...
!gsutil -u $GOOGLE_PROJECT du -h {MNT}/pgen/acaf_threshold.chr1.pgen {MNT}/pgen/acaf_threshold.chr22.pgen


## 2. Per-ancestry keep-lists from the S3 analytic parquet
`analytic_pooled.parquet` already carries `person_id · ancestry · PC1-16 · qc_pass` (+ phenotypes/covariates), so no separate ancestry re-load is needed.
Only ancestries clearing the S2 census floor (≥~1,000 analytic, ≥~100 cases) are exported: **EUR, AFR, AMR**.
AMR is exported **for the trans-ancestry meta only** (RUS: no standalone AMR arm). EAS/SAS/MID (120/40/40) below floor -> excluded; OTH excluded.

In [ ]:
import pandas as pd
df = pd.read_parquet(f"{AB}/analytic/analytic_pooled.parquet", columns=["person_id","ancestry","qc_pass"])
df = df[df.qc_pass==True].copy()
df["iid"] = df.person_id.astype("int64").astype(str)
df["anc"] = df.ancestry.astype(str).str.lower()
os.makedirs("/tmp/keep", exist_ok=True)
open("/tmp/keep/keep_cohort.txt","w").write("#IID\n"+"\n".join(df.iid)+"\n")   # rows never printed
print("COHORT N (qc_pass):", r20(len(df)))
for anc in ["eur","afr","amr"]:
    s = df[df.anc==anc]
    open(f"/tmp/keep/keep_{anc}.txt","w").write("#IID\n"+"\n".join(s.iid)+"\n")
    print(f"{anc.upper()} N:", r20(len(s)))
# validated: all 12,592 cohort IIDs are present in the ACAF .psam (535,662)


## 3. Genome-wide subset + variant QC (the ONE real pass)
For each autosome, in parallel (4 chromosomes at a time): **single gcsfuse read** to subset the 12,592 cohort samples from the mounted pgen,
then **local QC**: call-rate ≥0.95 (`--geno 0.05`), HWE p>1e-6 (`--hwe 1e-6`), **MAC≥20** (`--mac 20`). Multiallelics retained here (handled at export).
Reading 535,663 samples x ~44M variants (~1.4 TiB) to extract 12,592 takes ~2 h at ~200 MB/s aggregate; ~$0.50. Run as a background job.

In [ ]:
%%bash
# /tmp/run_geno.sh  (launched with: nohup bash /tmp/run_geno.sh > /tmp/logs/main.log 2>&1 &)
export MNT=$HOME/workspace/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold
mkdir -p /tmp/raw /tmp/cohort /tmp/logs
proc() {
  c=$1; L=/tmp/logs/chr${c}.log
  plink2 --pfile "$MNT/pgen/acaf_threshold.chr${c}" --keep /tmp/keep/keep_cohort.txt \
    --make-pgen --out /tmp/raw/chr${c} --threads 2 --memory 3500 > "$L" 2>&1        # single gcsfuse read
  plink2 --pfile /tmp/raw/chr${c} --geno 0.05 --hwe 1e-6 --mac 20 \
    --make-pgen --out /tmp/cohort/chr${c} --threads 2 --memory 3500 >> "$L" 2>&1    # local QC
  rm -f /tmp/raw/chr${c}.*
}
export -f proc
printf '%s\n' 1 22 2 21 3 20 4 19 5 18 6 17 7 16 8 15 9 14 10 13 11 12 | xargs -P 4 -I{} bash -c 'proc "$@"' _ {}


## 4. Per-ancestry export — Step-1 pruned predictors + Step-2 BGEN substrate
Merge the QC'd per-chr cohort pgens, then per ancestry (EUR, AFR, AMR): biallelic-only (`--max-alleles 2`, ACAF has multiallelics), ancestry **MAF≥1%**.
- **Step-1**: LD-prune (1 Mb window, r²<0.1) -> `step1_pruned_<anc>.{bed,bim,fam}` (REGENIE `--step 1 --bed`).
- **Step-2**: `--export bgen-1.2 bits=8` + `bgenix -index` -> `step2_<anc>.{bgen,sample,bgen.bgi}` (REGENIE `--step 2 --bgen/--sample`).
`.fam`/`.sample` use FID=0, IID=person_id. Files upload to `${WSB}/gwas/geno/{step1,step2}/`; aggregate `geno_manifest.tsv` written.

In [ ]:
%%bash
# /tmp/export_loop.sh  (excerpt; full script in _transfer)
plink2 --pmerge-list /tmp/mergelist.txt pfile --make-pgen --out /tmp/cohort_all --threads 4 --memory 20000
for anc in eur afr amr; do
  plink2 --pfile /tmp/cohort_all --keep /tmp/keep/keep_${anc}.txt --maf 0.01 --max-alleles 2 \
    --indep-pairwise 1000kb 0.1 --out /tmp/prune_${anc}
  plink2 --pfile /tmp/cohort_all --keep /tmp/keep/keep_${anc}.txt --maf 0.01 --max-alleles 2 \
    --extract /tmp/prune_${anc}.prune.in --make-bed --out /tmp/geno/step1/step1_pruned_${anc}
  plink2 --pfile /tmp/cohort_all --keep /tmp/keep/keep_${anc}.txt --maf 0.01 --max-alleles 2 \
    --export bgen-1.2 bits=8 --out /tmp/geno/step2/step2_${anc}
  bgenix -g /tmp/geno/step2/step2_${anc}.bgen -index
done
# gsutil cp step1_/step2_ + geno_manifest.tsv -> ${WSB}/gwas/geno/


## 5. Positive-control phenotype: arthroplasty-for-OA
Cases = TKA(27447) ∪ THA(27130) cohort members with a knee/hip OA dx (**ICD-10-CM M17\* / M16\***).
Controls = the 7 other-joint surgical cohorts **without** knee/hip OA. All genotyped, qc_pass -> stays inside the one export.
REGENIE pheno/covar written with FID=0, IID=person_id; covariates = age, age², sex, PC1–16.

In [ ]:
from google.cloud import bigquery
bq = bigquery.Client(project=GOOGLE_PROJECT)
def q(sql): return bq.query(sql.replace("{CDR}",WORKSPACE_CDR)).result().to_dataframe()
oa = q("SELECT DISTINCT co.person_id AS pid FROM `{CDR}.condition_occurrence` co "
       "JOIN `{CDR}.concept` c ON co.condition_source_concept_id=c.concept_id "
       "WHERE c.vocabulary_id='ICD10CM' AND (c.concept_code LIKE 'M17%' OR c.concept_code LIKE 'M16%')")
# case = cohort in {TKA,THA} & OA dx; control = other-joint cohort & no OA dx; write posctrl_pheno.tsv / posctrl_covar.tsv
# aggregate (round20): EUR 4,520 cases / 2,520 controls; AFR 620/380; AMR 400/420


## 6. Positive-control GWAS (REGENIE) + inflation / known-loci readout
REGENIE Step 1 (pruned predictors) -> Step 2 (Firth/`--approx`, `--minMAC 20`) within **EUR**, covariates age/age²/sex/PC1–16.
Expect recovery of **GDF5 (chr20), COL27A1 (chr9), COL11A1 (chr1), SLC39A8 (chr4)** with clean **λ_GC ~1.0–1.1**.

In [ ]:
%%bash
# /tmp/regenie.sh eur
regenie --step 1 --bed /tmp/geno/step1/step1_pruned_eur --phenoFile /tmp/posctrl_pheno.tsv \
  --covarFile /tmp/posctrl_covar.tsv --bt --bsize 1000 --lowmem --lowmem-prefix /tmp/rg_eur --out /tmp/step1_posctrl_eur
regenie --step 2 --bgen /tmp/geno/step2/step2_eur.bgen --sample /tmp/geno/step2/step2_eur.sample \
  --pred /tmp/step1_posctrl_eur_pred.list --phenoFile /tmp/posctrl_pheno.tsv --covarFile /tmp/posctrl_covar.tsv \
  --bt --firth --approx --pThresh 0.01 --minMAC 20 --bsize 400 --out /tmp/sumstats/pooled_eur
# -> gsutil cp .../pooled_eur.regenie  ${WSB}/gwas/sumstats/posctrl_oa/pooled_eur.regenie


In [ ]:
import numpy as np, scipy.stats as st, pandas as pd
ss = pd.read_csv("/tmp/sumstats/pooled_eur.regenie", sep=r"\s+")   # LOG10P column; P = 10**-LOG10P
p = np.power(10.0, -ss["LOG10P"].values)
lam = np.median(st.chi2.isf(p,1)) / st.chi2.ppf(0.5,1)
print("lambda_GC:", round(lam,3))                                   # expect ~1.0-1.1
# tabulate min-P in +/-250kb GRCh38 windows: GDF5 chr20:35.4Mb, COL27A1 chr9:114.1Mb, COL11A1 chr1:102.9Mb, SLC39A8 chr4:102.25Mb


## Definition of Done
Per-ancestry **step2** BGEN + pruned **step1** bed materialized in `${WSB}/gwas/geno/` for EUR/AFR/AMR · **no cluster launched** (Apps = single cheap VM) ·
positive control recovers GDF5/COL27A1/COL11A1/SLC39A8 with λ_GC ~1.0–1.1 · all model-visible output aggregate/≥20/MAC≥20 · sumstats stay in the bucket · spend logged.